# 03 — Knowledge graph theo thời gian

Việc 18 (plan §12). Notebook này cho thấy KG lấy từ Wikidata thay đổi thế nào khi chỉ giữ những fact đã đúng
**trước** mốc cắt (quyết định 0004 và 0006): fact nào có ngày, fact "tương lai" nào bị loại tại t1 và t2,
và vì sao điều đó quan trọng với các phim nổi tiếng như *Titanic* hay *Toy Story*.

> **Lưu ý:** Wikidata là một bản chụp gần đây, không phải tri thức có sẵn vào thời điểm rating (ngày chụp và
> khoảng thời gian của dữ liệu được in ở ô code đầu tiên). Mọi kết quả dùng KG phải nêu điều này.

**Nguồn:** chỉ đọc `data/raw/wikidata/facts.tsv`, đầu ra của `scripts/06_build_kg.py` trong
`data/processed/<dataset>/kg/<biến thể>/` (`kg_meta.json`, `dropped.csv`, `entities.csv`, `kg_t1.tsv`) và
`train.csv`. Hàm tính toán nằm trong `src/tkgat/analysis/kg.py` (test: `tests/test_analysis_kg.py`); quy tắc ngày
không được viết lại ở đây mà dùng `tkgat.kg.temporal`. Mọi con số trong phần nhận xét được in ra từ code.

**Ba biến thể KG:** `default` (lọc theo thời gian, giữ fact không có ngày nào), `drop_undated` (lọc theo thời gian,
bỏ fact không có ngày nào), `static` (giữ mọi fact như paper KGAT; chỉ dùng làm tham chiếu để đo rò rỉ qua KG).

In [ ]:
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from tkgat.analysis import kg

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# chart tokens (validated with the dataviz guide's palette script, light surface)
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA, NEUTRAL = "#2a78d6", "#eb6834", "#1baf7a", "#c3c2b7"
LABEL = {"ml-100k": "ML-100K", "ml-1m": "ML-1M"}
VARIANTS = ("default", "drop_undated", "static")
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": INK2,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "text.color": INK,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "lines.linewidth": 2,
        "legend.frameon": False,
        "figure.dpi": 110,
    }
)


def say(text):
    display(Markdown(text))


def pct(x, d=1):
    return f"{100 * x:.{d}f}%".replace(".", ",")


def num(x):
    return f"{int(x):,}".replace(",", ".")


def vn_day(s):
    return pd.Timestamp(s).strftime("%d/%m/%Y")


FACTS = pd.read_csv(
    ROOT / "data/raw/wikidata/facts.tsv", sep="\t", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE
)
DATA = {}
for name in LABEL:
    proc = ROOT / "data/processed" / name
    metas = {v: json.loads((proc / "kg" / v / "kg_meta.json").read_text(encoding="utf-8")) for v in VARIANTS}
    DATA[name] = {
        "metas": metas,
        "meta": metas["default"],
        "dropped": pd.read_csv(proc / "kg/default/dropped.csv", dtype=str),
        "entities": pd.read_csv(proc / "kg/static/entities.csv"),
        "static_t1": pd.read_csv(proc / "kg/static/kg_t1.tsv", sep="\t"),
        "triples": {
            (v, c): pd.read_csv(proc / f"kg/{v}/kg_{c}.tsv", sep="\t")
            for v in ("static", "default")
            for c in ("t1", "t2")
        },
        "train": pd.read_csv(proc / "train.csv"),
        "val": pd.read_csv(proc / "val.csv"),
        "test": pd.read_csv(proc / "test.csv"),
    }
for name, d in DATA.items():
    m = d["meta"]
    print(
        f"{LABEL[name]}: t1 = {m['cutoffs']['t1']['day_utc']}, t2 = {m['cutoffs']['t2']['day_utc']}, "
        f"snapshot {m['wikidata_snapshot']}, facts md5 {m['inputs']['facts_md5'][:8]}"
    )

LINKS = pd.read_csv(ROOT / "data/processed/links/ml100k_links.csv", dtype={"freebase_id": str})
TITLE = dict(zip(LINKS["freebase_id"], LINKS["title"], strict=True))
years = [pd.to_datetime(d[p]["ts"], unit="s").dt.year for d in DATA.values() for p in ("train", "val", "test")]
say(
    f"> **Lưu ý:** Wikidata là bản chụp ngày {vn_day(DATA['ml-100k']['meta']['wikidata_snapshot'])}, không phải tri "
    f"thức có sẵn trong giai đoạn {min(y.min() for y in years)}–{max(y.max() for y in years)} của các tương tác."
)

## 1. Độ phủ và loại ngày theo quan hệ

In [ ]:
kinds = {n: kg.relation_date_kinds(d["meta"]) for n, d in DATA.items()}
cover = pd.DataFrame(
    {
        LABEL[n]: {
            "item train": d["meta"]["n_items"],
            "item có liên kết Freebase": d["meta"]["n_items_linked"],
            "item có fact": d["meta"]["n_items_with_facts"]["t1"],
            "số fact của item train": int(kinds[n]["facts"].sum()),
            "số quan hệ": d["meta"]["n_relations"],
        }
        for n, d in DATA.items()
    }
)
display(cover)
display(pd.concat({LABEL[n]: k for n, k in kinds.items()}, axis=1).round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), sharey=True, layout="constrained")
order = kinds["ml-100k"].index[::-1]
parts = [
    ("statement_dated", "có ngày statement", BLUE),
    ("tail_date_only", "chỉ có ngày của đích", AQUA),
    ("no_date", "không có ngày nào", NEUTRAL),
]
for ax, name in zip(axes, DATA, strict=True):
    k = kinds[name].reindex(order)
    left = np.zeros(len(k))
    for col, lab, color in parts:
        ax.barh(k.index, k[col], left=left, color=color, label=lab, height=0.7, edgecolor=SURFACE, linewidth=2)
        left += k[col].to_numpy()
    ax.set_xlim(0, 1)
    ax.set_title(f"{LABEL[name]}: loại ngày của fact")
    ax.grid(axis="y", visible=False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=3)
plt.show()

k100 = kinds["ml-100k"]
stmt = {n: float((k["facts"] * k["statement_dated"]).sum() / k["facts"].sum()) for n, k in kinds.items()}
stmt_counts = k100["facts"] * k100["statement_dated"]
share_two = float(stmt_counts[list(kg.AWARD_RELATIONS)].sum() / stmt_counts.sum())
nodate = {n: float((k["facts"] * k["no_date"]).sum() / k["facts"].sum()) for n, k in kinds.items()}
top_stmt = k100["statement_dated"].sort_values(ascending=False)
say(f"""
- {num(cover.loc["item có fact", LABEL["ml-100k"]])} trên {num(cover.loc["item train", LABEL["ml-100k"]])} item
  train của
  ML-100K và {num(cover.loc["item có fact", LABEL["ml-1m"]])} trên
  {num(cover.loc["item train", LABEL["ml-1m"]])} của ML-1M có
  ít nhất một fact.
- Chỉ {pct(stmt["ml-100k"])} fact (ML-100K) có **ngày của chính statement**, và {pct(share_two)} trong số đó
  thuộc hai
  quan hệ `{top_stmt.index[0]}` ({pct(top_stmt.iloc[0])} fact của quan hệ này có ngày) và `{top_stmt.index[1]}`
  ({pct(top_stmt.iloc[1])}). Đây cũng là hai quan hệ có thể làm rò rỉ tương lai, vì giải thưởng được trao
  **sau** khi phim
  ra mắt.
- Phần lớn fact còn lại chỉ có ngày tồn tại của đích (năm sinh của diễn viên, năm thành lập của hãng): ngày này
  cho biết
  đích đã tồn tại trước mốc hay chưa, chứ không cho biết khi nào liên kết trở thành đúng.
  {pct(nodate["ml-100k"])} / {pct(nodate["ml-1m"])} fact không có ngày nào, chủ yếu `genre`,
  `original_language`, `main_subject`.
""")

## 2. Độ chính xác của ngày (giới hạn A của quyết định 0006)

In [ ]:
by_prec, jan1 = kg.date_precision(FACTS)
display(by_prec.rename("số point_in_time").to_frame())
display(jan1.assign(share_jan1=jan1["share_jan1"].map(pct)))
year_only = by_prec.get("year", 0) / by_prec.sum()
say(f"""
- Chỉ `point_in_time` có trường độ chính xác: {pct(year_only)} trong {num(by_prec.sum())} giá trị chỉ biết
  **năm**,
  được hiểu là ngày 31/12 của năm đó (thận trọng: không bao giờ để lọt fact tương lai).
- Các cột ngày khác bị mất độ chính xác khi tải về; Wikidata ghi ngày chỉ có năm thành `YYYY-01-01`. Tỷ lệ kết
  thúc bằng
  `-01-01` là {pct(jan1.loc["film_publication", "share_jan1"])} ở ngày phát hành phim và
  {pct(jan1.loc["tail_inception", "share_jan1"])}
  ở ngày thành lập của đích, nhưng chỉ {pct(jan1.loc["tail_birth", "share_jan1"])} ở ngày sinh (ngày sinh
  thường đủ ngày).
  Quyết định 0006 A đọc các ngày này là năm-only (31/12): một fact thật sự xảy ra ngày 1/1 sẽ bị đặt muộn hơn.
  Đây là giới
  hạn đã biết, ghi trong `kg_meta.json`.
""")

## 3. Ba biến thể KG tại t1 và t2

In [ ]:
vt = pd.concat({LABEL[n]: kg.variant_triples(d["metas"]) for n, d in DATA.items()}, names=["dataset", "variant"])
display(vt)


def tri(name, variant, cut):
    return int(vt.loc[(LABEL[name], variant), cut])


only_t2 = {n: d["meta"]["n_entities_only_in_t2"] for n, d in DATA.items()}
grow_1m = tri("ml-1m", "default", "t2") - tri("ml-1m", "default", "t1")
why_1m = " vì không có fact nào mang ngày nằm giữa hai mốc" if grow_1m == 0 else ""
say(f"""
- **static − default tại t1** là số fact "tương lai" bị lọc:
  {num(tri("ml-100k", "static", "t1") - tri("ml-100k", "default", "t1"))}
  trên {num(tri("ml-100k", "static", "t1"))}
  ({pct(1 - tri("ml-100k", "default", "t1") / tri("ml-100k", "static", "t1"))}) ở ML-100K và
  {num(tri("ml-1m", "static", "t1") - tri("ml-1m", "default", "t1"))} trên {num(tri("ml-1m", "static", "t1"))}
  ({pct(1 - tri("ml-1m", "default", "t1") / tri("ml-1m", "static", "t1"))}) ở ML-1M. Tỷ lệ nhỏ, nhưng mục 5 cho
  thấy chúng
  rơi vào đâu.
- **default − drop_undated** là số fact không có ngày nào:
  {num(tri("ml-100k", "default", "t1") - tri("ml-100k", "drop_undated", "t1"))}
  (ML-100K) và {num(tri("ml-1m", "default", "t1") - tri("ml-1m", "drop_undated", "t1"))} (ML-1M).
- Từ t1 đến t2, KG default của ML-100K tăng
  {num(tri("ml-100k", "default", "t2") - tri("ml-100k", "default", "t1"))} fact; của ML-1M
  tăng {num(grow_1m)} fact{why_1m}.
  Số thực thể chỉ xuất hiện ở KG t2: {only_t2["ml-100k"]} (ML-100K) và {only_t2["ml-1m"]} (ML-1M).
""")

## 4. Fact "tương lai" bị loại, theo quan hệ

In [ ]:
drops = {n: kg.dropped_by_relation(d["meta"]) for n, d in DATA.items()}
display(drops["ml-100k"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, name in zip(axes, DATA, strict=True):
    t = drops[name]
    t = t[(t["future_statement_t1"] + t["future_tail_t1"]) > 0]
    t = t.assign(total=t["future_statement_t1"] + t["future_tail_t1"]).sort_values("total")
    ax.barh(
        t.index, t["future_tail_t1"], color=BLUE, label="đích chưa tồn tại", height=0.65, edgecolor=SURFACE, linewidth=2
    )
    ax.barh(
        t.index,
        t["future_statement_t1"],
        left=t["future_tail_t1"],
        color=ORANGE,
        label="ngày statement sau mốc",
        height=0.65,
        edgecolor=SURFACE,
        linewidth=2,
    )
    for y, v in enumerate(t["total"]):
        ax.text(v, y, f" {v}", va="center", color=INK2, fontsize=8)
    ax.set_title(f"{LABEL[name]}: fact bị loại tại t1")
    ax.set_xlabel("số fact")
    ax.grid(axis="y", visible=False)
axes[1].legend(loc="lower right")
fig.tight_layout()
plt.show()

d100 = drops["ml-100k"]
aw_t1 = int(d100.loc["award_received", "future_statement_t1"])
aw_t2 = int(d100.loc["award_received", "future_statement_t2"])
biggest_tail = d100["future_tail_t1"].idxmax()
dist_examples = DATA["ml-100k"]["dropped"]
dist_examples = dist_examples[(dist_examples["relation"] == biggest_tail) & (dist_examples["cutoff"] == "t1")]
top_labels = dist_examples["tail_label"].value_counts().head(3)
n_tail, n_stmt = int(d100["future_tail_t1"].sum()), int(d100["future_statement_t1"].sum())
n_aw_stmt = int(d100.loc[list(kg.AWARD_RELATIONS), "future_statement_t1"].sum())
dr = DATA["ml-100k"]["dropped"]
aw_key = ["freebase_id", "tail"]
aw_dropped = {
    c: dr[(dr["cutoff"] == c) & (dr["relation"] == "award_received") & (dr["reason"] == "future_statement")]
    for c in ("t1", "t2")
}
between = aw_dropped["t1"].merge(aw_dropped["t2"][aw_key], on=aw_key, how="left", indicator=True)
between = between[between["_merge"] == "left_only"]
main_day = between["point_in_time"].value_counts()
n_oscar_between = int(between["tail_label"].str.startswith("Academy Award").sum())
say(f"""
- Có {num(n_tail)} fact (ML-100K) bị loại vì "đích chưa tồn tại" và {num(n_stmt)} fact vì "ngày statement sau
  mốc". Với lý
  do thứ nhất, nhiều nhất là `{biggest_tail}` ({d100.loc[biggest_tail, "future_tail_t1"]} fact),
  ví dụ {", ".join(f"{lab} ({c} phim)" for lab, c in top_labels.items())}, là những đơn vị phát hành ra đời
  nhiều năm sau
  khi rating được nhập. Phần tiếp theo (`followed_by`) ra mắt sau mốc cũng bị loại theo lý do này.
- Với lý do "ngày statement sau mốc", {pct(n_aw_stmt / n_stmt)} là giải thưởng và đề cử. Ở ML-100K, số fact
  `award_received` bị loại giảm từ
  {aw_t1} tại t1 xuống {aw_t2} tại t2: {len(between)} giải được trao **giữa** hai mốc ({main_day.iloc[0]} giải
  cùng ngày
  {vn_day(main_day.index[0])}, trong đó {n_oscar_between} giải Oscar), nên có trong KG dùng cho test nhưng
  không có trong KG
  dùng cho train/val.
""")

## 5. Fact bị loại rơi vào phim nổi tiếng

In [ ]:
conc = {}
for name, d in DATA.items():
    counts = kg.item_fact_counts(d["static_t1"], d["entities"])
    dropped_t1 = d["dropped"][d["dropped"]["cutoff"] == "t1"]
    conc[name] = kg.popular_concentration(counts, dropped_t1, d["train"], frac=0.1)
conc_t = pd.DataFrame({LABEL[n]: c for n, c in conc.items()})
display(conc_t)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
x = np.arange(len(conc))
w = 0.36
a = [c["share_of_all_facts"] for c in conc.values()]
b = [c["share_of_future_award_facts"] for c in conc.values()]
ax.bar(x - w / 2, a, w, color=BLUE, label="mọi fact", edgecolor=SURFACE, linewidth=2)
ax.bar(
    x + w / 2, b, w, color=ORANGE, label="giải thưởng/đề cử tương lai bị loại tại t1", edgecolor=SURFACE, linewidth=2
)
for xs, vals in ((x - w / 2, a), (x + w / 2, b)):
    for xi, v in zip(xs, vals, strict=True):
        ax.text(xi, v + 0.01, pct(v, 0), ha="center", color=INK, fontsize=9)
ax.set_xticks(x, [LABEL[n] for n in conc])
ax.set_ylim(0, max(b) * 1.3)
ax.set_ylabel("tỷ lệ thuộc 10% item phổ biến nhất")
ax.set_title("Phần của 10% item phổ biến nhất (train)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()

c100, c1 = conc["ml-100k"], conc["ml-1m"]
say(f"""
- 10% item được rate nhiều nhất trong train ({c100["top_items"]} phim ở ML-100K, {c1["top_items"]} ở ML-1M) giữ
  {pct(c100["share_of_all_facts"])} / {pct(c1["share_of_all_facts"])} tổng số fact, nhưng chiếm
  **{pct(c100["share_of_future_award_facts"])} / {pct(c1["share_of_future_award_facts"])}** số fact giải
  thưởng/đề cử "tương lai"
  bị loại tại t1 ({c100["future_award_facts"]} fact trên {c100["films_losing_award_facts"]} phim ở ML-100K;
  {c1["future_award_facts"]} fact trên {c1["films_losing_award_facts"]} phim ở ML-1M).
- Vì vậy dù chỉ {pct(1 - tri("ml-100k", "default", "t1") / tri("ml-100k", "static", "t1"))} (ML-100K) và
  {pct(1 - tri("ml-1m", "default", "t1") / tri("ml-1m", "static", "t1"))} (ML-1M) fact bị lọc tại t1, một KG tĩnh
  (biến thể `static`) cho mô hình biết trước thành công của
  chính những
  phim đang được nhiều người xem: đây là kênh rò rỉ mà biến thể `static` dùng để đo.
""")

## 6. Tương tác val/test rơi vào item có KG bị lọc

Một item "bị lọc" là item có số triple khác nhau giữa KG `static` và KG `default` **tại cùng một mốc**, tức mất ít nhất
một fact vì quy tắc thời gian. Val được chấm với KG tại t1, test với KG tại t2.

In [ ]:
aff_rows, affected = {}, {}
for name, d in DATA.items():
    for cut, part in (("t1", "val"), ("t2", "test")):
        items = kg.items_with_filtered_facts(
            d["triples"][("static", cut)], d["triples"][("default", cut)], d["entities"]
        )
        affected[(name, part)] = items
        pos = d[part]
        aff_rows[(LABEL[name], f"{part} (KG tại {cut})")] = {
            "item bị lọc": len(items),
            "tỷ lệ item train": len(items) / d["meta"]["n_items"],
            "tỷ lệ positive": kg.positive_share(pos, items),
            "user có ít nhất 1 positive trên item bị lọc": int(pos.loc[pos["item"].isin(items), "user"].nunique()),
            "số user": int(pos["user"].nunique()),
        }
aff = pd.DataFrame(aff_rows).T
display(aff)


def share(name, part):
    return aff.loc[(LABEL[name], f"{part} (KG tại {'t1' if part == 'val' else 't2'})"), "tỷ lệ positive"]


def n_aff(name, part):
    return int(aff.loc[(LABEL[name], f"{part} (KG tại {'t1' if part == 'val' else 't2'})"), "item bị lọc"])


say(f"""
- Chỉ {pct(1 - tri("ml-100k", "default", "t1") / tri("ml-100k", "static", "t1"))} (ML-100K) và
  {pct(1 - tri("ml-1m", "default", "t1") / tri("ml-1m", "static", "t1"))} (ML-1M) fact bị lọc tại t1, nhưng
  {n_aff("ml-100k", "val")}
  item của ML-100K và {n_aff("ml-1m", "val")} item của ML-1M mất ít nhất một fact, và các item này nhận
  **{pct(share("ml-100k", "val"))} positive val / {pct(share("ml-100k", "test"))} positive test** ở ML-100K,
  **{pct(share("ml-1m", "val"))} / {pct(share("ml-1m", "test"))}** ở ML-1M.
- Nghĩa là từ {pct(aff["tỷ lệ positive"].min(), 0)} đến {pct(aff["tỷ lệ positive"].max(), 0)} số tương tác dùng
  để chọn
  mô hình và chấm điểm nằm trên item mà KG tĩnh chứa thông tin tương lai: khác biệt giữa `static` và `default`
  có thể ảnh hưởng thật đến Recall@20, không chỉ là vài fact lẻ.
  Mức ảnh hưởng thực tế sẽ đo khi chạy KGAT với hai biến thể.
""")

## 7. Ví dụ: *Titanic*

In [ ]:
TITANIC = "m.0dr_4"
t1d, t2d = (pd.Timestamp(DATA["ml-100k"]["meta"]["cutoffs"][c]["day_utc"]) for c in ("t1", "t2"))
tit = kg.film_fact_status(FACTS[FACTS["freebase_id"] == TITANIC], DATA["ml-100k"]["dropped"])
aw = tit[tit["relation"].isin(kg.AWARD_RELATIONS)]
display(aw.groupby(["relation", "status"]).size().unstack(fill_value=0))

STATUS = {
    "kept_t1": ("có từ t1", BLUE),
    "added_t2": ("chỉ có từ t2", ORANGE),
    "dropped_both": ("bị loại ở cả hai", MUTED),
}
dated = aw.dropna(subset=["statement_date"]).copy()
dated["date"] = pd.to_datetime(dated["statement_date"].astype(str))
window = dated[(dated["date"] >= "1997-12-01") & (dated["date"] <= "1999-06-30")]
fig, ax = plt.subplots(figsize=(9, 2.8))
rows = {"nominated_for": 0, "award_received": 1}
rng = np.random.default_rng(0)  # vertical jitter only, so that facts on the same day stay visible
for st, (lab, color) in STATUS.items():
    s = window[window["status"] == st]
    y = s["relation"].map(rows) + rng.uniform(-0.18, 0.18, len(s))
    ax.scatter(
        s["date"],
        y,
        s=40,
        color=color,
        label=f"{lab} ({len(s)} trong khung)",
        edgecolor=SURFACE,
        linewidth=1.5,
        zorder=3,
    )
for t, lab, ha in ((t1d, "t1 ", "right"), (t2d, " t2", "left")):
    ax.axvline(t, color=INK2, lw=1, ls="--")
    ax.text(t, 1.45, lab, color=INK2, ha=ha, va="top", fontsize=9)
ax.set_yticks(list(rows.values()), ["đề cử", "giải thưởng"])
ax.set_ylim(-0.5, 1.5)
ax.set_title("Titanic: ngày statement của đề cử và giải thưởng (12/1997–06/1999)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

added = aw[aw["status"] == "added_t2"]
oscars = added["tail_label"].str.startswith("Academy Award").sum()
dates_added = sorted({str(x) for x in added["statement_date"].dropna()})
lost_t1 = int((tit["status"] != "kept_t1").sum())
both = aw[aw["status"] == "dropped_both"]
both_dates = both["statement_date"].dropna().astype(str)
both_year_only = int(both_dates.str.endswith("-12-31").sum())
year_only_days = " hoặc ".join(f"31/12/{y}" for y in sorted({d[:4] for d in both_dates if d.endswith("-12-31")}))
kept_undated = aw[aw["statement_date"].isna() & (aw["status"] == "kept_t1")]
nom_oscar = aw[
    (aw["relation"] == "nominated_for") & (aw["status"] == "kept_t1") & aw["tail_label"].str.startswith("Academy Award")
]
nom_day = nom_oscar["statement_date"].astype(str).value_counts().index[0]
if oscars == len(added):
    awards_phrase = f"{len(added)} giải thưởng, **đều là giải Oscar**"
else:
    awards_phrase = f"{len(added)} giải thưởng, trong đó **{oscars} giải Oscar**"
say(f"""
- *{TITLE[TITANIC]}* có {len(tit)} fact; KG tại t1 thiếu **{lost_t1}** fact trong số đó. {awards_phrase},
  được trao ngày {", ".join(vn_day(x) for x in dates_added)}, tức sau t1
  ({vn_day(t1d)}) nhưng trước t2
  ({vn_day(t2d)}): chúng không có trong KG train/val nhưng có trong KG test.
- {len(both)} giải/đề cử bị loại ở cả hai mốc; {both_year_only} trong số đó chỉ có năm
  (đọc là {year_only_days} theo
  quy tắc 0006),
  phần còn lại có ngày statement hoặc ngày ra đời của giải sau t2.
- {len(nom_oscar)} đề cử Oscar ({vn_day(nom_day)}) có ngay từ t1 vì xảy ra trước mốc. {len(kept_undated)}
  giải/đề cử không
  có ngày statement (ví dụ {kept_undated["tail_label"].iloc[0]}) vẫn được giữ từ t1 trong biến thể `default` vì
  giải đã tồn
  tại từ trước, dù không biết phim nhận giải khi nào: đây là giới hạn của quy tắc mặc định.
""")

## 8. Ví dụ: *Toy Story* và phần tiếp theo

In [ ]:
TOY_STORY = "m.0dyb1"
toy = kg.film_fact_status(FACTS[FACTS["freebase_id"] == TOY_STORY], DATA["ml-100k"]["dropped"])
toy_lost = toy[toy["status"] != "kept_t1"]
display(toy_lost[["relation", "tail_label", "statement_date", "tail_date", "status", "reason_t1"]])
seq = toy_lost[toy_lost["relation"] == "followed_by"].iloc[0]
say(f"""
- *{TITLE[TOY_STORY]}* có {len(toy)} fact; {len(toy_lost)} fact không có trong KG tại t1. Liên kết `followed_by →
  {seq["tail_label"]}`
  bị loại ở cả hai mốc: phần tiếp theo ra mắt ngày {vn_day(seq["tail_date"])}, sau toàn bộ dữ liệu ML-100K. Một
  KG tĩnh sẽ
  cho mô hình biết trước rằng phim này sẽ có phần tiếp theo.
- Các fact còn lại của *{TITLE[TOY_STORY]}* (đạo diễn, diễn viên, thể loại) đã đúng từ trước nên được giữ.
""")

## 9. Giới hạn đã biết

In [ ]:
for line in DATA["ml-100k"]["meta"]["known_limitations"]:
    print("-", line)
no_fact = {n: d["meta"]["n_items"] - d["meta"]["n_items_with_facts"]["t1"] for n, d in DATA.items()}
say(f"""
- {no_fact["ml-100k"]} item train của ML-100K và {no_fact["ml-1m"]} của ML-1M không có fact nào (không liên kết
  được hoặc Wikidata
  không có dữ liệu); mô hình KG phải xử lý item không có cạnh KG.
- Thực thể chỉ có ở KG t2 ({only_t2["ml-100k"]} / {only_t2["ml-1m"]}) chưa được xử lý gì; sẽ quyết định ở giai
  đoạn KGAT.
""")

## 10. Tóm tắt

In [ ]:
say(f"""
1. Chỉ {pct(stmt["ml-100k"])} fact có ngày statement, gần như chỉ giải thưởng và đề cử; phần lớn fact chỉ biết
   đích đã tồn tại từ khi nào.
2. Lọc theo thời gian bỏ {pct(1 - tri("ml-100k", "default", "t1") / tri("ml-100k", "static", "t1"))} fact ở
   ML-100K và
   {pct(1 - tri("ml-1m", "default", "t1") / tri("ml-1m", "static", "t1"))} ở ML-1M tại t1.
3. Những fact bị bỏ tập trung vào phim nổi tiếng: 10% item phổ biến nhất chiếm
   {pct(c100["share_of_future_award_facts"])} /
   {pct(c1["share_of_future_award_facts"])} fact giải thưởng/đề cử tương lai, so với
   {pct(c100["share_of_all_facts"])} /
   {pct(c1["share_of_all_facts"])} tổng số fact.
4. Các item mất ít nhất một fact nhận {pct(share("ml-100k", "val"))} / {pct(share("ml-100k", "test"))} positive
   val/test ở ML-100K
   và {pct(share("ml-1m", "val"))} / {pct(share("ml-1m", "test"))} ở ML-1M.
5. *Titanic* thiếu {lost_t1} fact tại t1; {oscars} giải Oscar chỉ xuất hiện trong KG test (t2).
6. Biến thể `static` (như paper KGAT) chứa các fact này và chỉ dùng làm tham chiếu để đo rò rỉ qua KG.
""")